# MLP Weight Structure

Analyze MLP weight matrices: spectral properties, neuron norms,
in-out alignment, cross-layer similarity, and unembed alignment.

## Why This Matters

MLP weight structure provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_weight_structure import (
    mlp_weight_spectrum, mlp_neuron_norms,
    mlp_in_out_alignment, mlp_cross_layer_similarity,
    mlp_unembed_alignment,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Weight Spectrum

Singular value decomposition of W_in and W_out reveals effective rank.

In [ ]:
for layer in range(4):
    result = mlp_weight_spectrum(model, layer=layer)
    print(f"Layer {layer}: W_in rank={result['W_in_effective_rank']:.2f}, "
          f"W_out rank={result['W_out_effective_rank']:.2f}, "
          f"top_sv_in={result['W_in_top_sv_fraction']:.3f}, "
          f"top_sv_out={result['W_out_top_sv_fraction']:.3f}")

## Neuron Norms

Which neurons have the largest combined input-output norms?

In [ ]:
result = mlp_neuron_norms(model, layer=0, top_k=8)
print(f"d_mlp={result['d_mlp']}, mean_in={result['mean_in_norm']:.4f}, "
      f"mean_out={result['mean_out_norm']:.4f}\n")
for n in result['top_neurons']:
    bar = '█' * int(min(n['combined_norm'] * 10, 20))
    print(f"  Neuron {n['neuron']:3d}: in={n['in_norm']:.4f}, "
          f"out={n['out_norm']:.4f}, combined={n['combined_norm']:.4f} {bar}")

## In-Out Alignment

How aligned is each neuron's input direction with its output direction?
High alignment means the neuron reads and writes in similar subspaces.

In [ ]:
for layer in range(4):
    result = mlp_in_out_alignment(model, layer=layer)
    print(f"Layer {layer}: mean_align={result['mean_alignment']:.4f}, "
          f"aligned={result['n_aligned']}/{result['n_total']} "
          f"({result['fraction_aligned']:.1%})")

## Cross-Layer Similarity

Are MLP weights similar across layers, suggesting redundant computation?

In [ ]:
result = mlp_cross_layer_similarity(model)
for p in result['pairs']:
    print(f"  Layer {p['layer_a']} vs {p['layer_b']}: "
          f"W_in_sim={p['W_in_similarity']:.4f}, "
          f"W_out_sim={p['W_out_similarity']:.4f}")

## Unembed Alignment

Which vocabulary tokens does each MLP neuron most promote or suppress
through its output direction projected through W_U?

In [ ]:
result = mlp_unembed_alignment(model, layer=0, top_k=8)
for n in result['per_neuron']:
    print(f"  Neuron {n['neuron']:3d}: norm={n['output_norm']:.4f}, "
          f"promotes tok {n['top_promoted_token']:3d} ({n['top_promoted_logit']:.4f}), "
          f"suppresses tok {n['top_suppressed_token']:3d} ({n['top_suppressed_logit']:.4f})")